# Linear Mixed Model Analysis

This notebook implements a comprehensive linear mixed model analysis to examine group differences in brain measurements while accounting for various covariates and random effects.

## Overview

The analysis pipeline includes:

1. **Data Preparation**
   - Loads deviation scores from normative modeling
   - Merges with demographic data
   - Restructures data for mixed model analysis

2. **Model Fitting**
   - Implements linear mixed models for each brain region
   - Includes fixed effects:
     - Group (HC，IBS<sub>ROME III</sub>, IBS<sub>Both</sub>, IBS<sub>Diagnosed</sub>)
     - Sex
     - Hemisphere
   - Includes random effects:
     - Subject

3. **Statistical Analysis**
   - Tests main effects and interactions
   - Performs post-hoc analyses
   - Applies multiple comparison corrections

4. **Visualization**
   - Generates brain region plots
   - Creates summary tables
   - Visualizes significant findings

## Usage Instructions

1. Set the correct paths in the first code cell:
   - `root_dir`: Path to project root
   - `docu_dir`: Path to documentation files
   - `data_dir`: Path to preprocessed data
   - `model_dir`: Path to normative model outputs
   - `out_dir`: Path for analysis results

2. Run cells sequentially to:
   - Prepare data for analysis
   - Fit mixed models
   - Perform post-hoc tests
   - Generate visualizations

## Key Parameters
- `perm`: Should be align with model training
- `ROI_list`: List of brain regions to analyze
- `vmin`, `vmax`: Visualization parameters
- `cmp`: Color map for brain plots

## Output Files

The analysis generates:
- Statistical results tables
- Brain region visualizations
- Post-hoc analysis results
- Summary statistics

In [1]:
import os
import pandas as pd
from utils_norm.utils_analyses import prepare_data_for_long_structure, run_mixed_effects_analysis

perm = 1
root_dir = 'Abosolute path to this project'
docu_dir = os.path.join(root_dir, '1_document')
data_dir = os.path.join(root_dir, '3_rerun_whole_work', '1_data_cleaned')
model_dir = os.path.join(root_dir, '3_rerun_whole_work', '2_models_sMRI')
out_dir = os.path.join(root_dir, '3_rerun_whole_work', '3_LMM')
os.makedirs(out_dir,exist_ok=True)

# Linear mixed model analysis

## run LMM

In [2]:
HC_cov = pd.read_csv(os.path.join(data_dir, 'HC_MRI_age.csv'), usecols=['eid', '31-0.0'])
IBS_cov = pd.read_csv(os.path.join(data_dir, 'IBS_All_MRI_age.csv'), usecols=['eid', '31-0.0', 'Group'])
HC_cov.rename(columns={'31-0.0':'sex'}, inplace=True)
IBS_cov.rename(columns={'31-0.0':'sex'}, inplace=True)
HC_cov['Group'] = 0
ROI_list = pd.read_csv(os.path.join(docu_dir, 'ROI_IBS.csv'))['ROI'].tolist()

In [3]:
for file_name in ['CT', 'SA', 'CV']:
    results_dir = os.path.join(out_dir, file_name)
    os.makedirs(results_dir,exist_ok=True)
    deviation_dir = os.path.join(model_dir, file_name+'_age_45_85', 'perm_'+str(perm))
    HC_deviation = pd.read_csv(os.path.join(deviation_dir, 'HC_deviation.csv'))
    IBS_deviation = pd.read_csv(os.path.join(deviation_dir, 'Patient_deviation_IBS.csv'))

    HC_data = HC_deviation.merge(HC_cov, on='eid', how='left')
    IBS_data = IBS_deviation.merge(IBS_cov, on='eid', how='left')
    data_long = prepare_data_for_long_structure(HC_data, IBS_data, ROI_list)
    run_mixed_effects_analysis(data_long, results_dir)

## Results visualization

In [4]:
from utils_norm.utils_visual import plot_brain_mixedlm

In [ ]:
vmin = 0
vmax = 0.05
cmp = 'black_blue_r'
for modality in ['CT', 'SA', 'CV']:
    results_dir = os.path.join(out_dir, modality)
    file_list = ['main_effect_group', 'interaction_group_sex', 'interaction_group_hemisphere', 'interaction_group_sex_hemisphere']
    for file_name in file_list:
        for external_group in range(1, 4):
            result_table = pd.read_csv(os.path.join(results_dir, file_name+'_'+str(external_group)+'.csv'))
            result_table.rename(columns={'roi': 'BrainArea'}, inplace=True)
            # plot brain regions with un corrected sig. p value, save brain regions with sig. p value after FDR correction in the log.
            plot_brain_mixedlm(result_table, 'P>|z|', 'FDR_P', docu_dir, results_dir, results_name='group_'+str(external_group)+'_'+file_name, log_name='mixedlm_analysis_group_'+str(external_group)+'_'+file_name, vmin=vmin, vmax=vmax, color_map=cmp)

# Post hoc analysis

## Right-Left t-test

In [2]:
from statsmodels.stats.multitest import multipletests
from utils_norm.utils_analyses import t_test_levene, equivalence_test

In [3]:
test_list = {'SA':{'Group':[1,1,3], 'ROI':['S_precentral_inf_part', 'G_and_S_cingul_Mid_Post','S_intrapariet_and_P_trans'], 'Sex':[None,None,None]}, 
             'CV':{'Group':[3], 'ROI':['S_intrapariet_and_P_trans'], 'Sex':[None]}}

In [4]:
HC_cov = pd.read_csv(os.path.join(data_dir, 'HC_MRI_age.csv'), usecols=['eid', '31-0.0'])
IBS_cov = pd.read_csv(os.path.join(data_dir, 'IBS_All_MRI_age.csv'), usecols=['eid', '31-0.0', 'Group'])
HC_cov.rename(columns={'31-0.0':'sex'}, inplace=True)
IBS_cov.rename(columns={'31-0.0':'sex'}, inplace=True)

t_test_dataframe = []
t_test_table = []
for file_name in ['SA', 'CV']:
    deviation_dir = os.path.join(model_dir, file_name+'_age_45_85', 'perm_'+str(perm))
    HC_deviation = pd.read_csv(os.path.join(deviation_dir, 'HC_deviation.csv'))
    IBS_deviation = pd.read_csv(os.path.join(deviation_dir, 'Patient_deviation_IBS.csv'))

    HC_data = HC_deviation.merge(HC_cov, on='eid', how='left')
    IBS_data = IBS_deviation.merge(IBS_cov, on='eid', how='left')
    for group, idp_str in zip(test_list[file_name]['Group'], test_list[file_name]['ROI']):
        HC_data[idp_str+'_rh-lh'] = HC_data['rh_'+ idp_str] - HC_data['lh_'+ idp_str]
        IBS_data[idp_str+'_rh-lh'] = IBS_data['rh_'+ idp_str] - IBS_data['lh_'+ idp_str]
        IBS_data_sub = IBS_data[IBS_data['Group']==group]
        IBS_data_sub.reset_index(inplace=True)
        for HC_sex in range(0,2):
            HC_data_sex = HC_data[HC_data['sex']==HC_sex]
            for IBS_sex in range(0,2):
                IBS_data_sex = IBS_data_sub[IBS_data_sub['sex']==IBS_sex]

                mean_hc_left = HC_data_sex['lh_'+ idp_str].mean()
                std_hc_left = HC_data_sex['lh_'+ idp_str].std()
                mean_hc_right = HC_data_sex['rh_'+ idp_str].mean()
                std_hc_right = HC_data_sex['rh_'+ idp_str].std()
                
                mean_ibs_left = IBS_data_sex['lh_'+ idp_str].mean()
                std_ibs_left = IBS_data_sex['lh_'+ idp_str].std()
                mean_ibs_right = IBS_data_sex['rh_'+ idp_str].mean()
                std_ibs_right = IBS_data_sex['rh_'+ idp_str].std()

                mean_hc_diff = HC_data_sex[idp_str+'_rh-lh'].mean()
                std_hc_diff = HC_data_sex[idp_str+'_rh-lh'].std()
                mean_ibs_diff = IBS_data_sex[idp_str+'_rh-lh'].mean()
                std_ibs_diff = IBS_data_sex[idp_str+'_rh-lh'].std()

                t_stat, p_value, df = t_test_levene(HC_data_sex[idp_str+'_rh-lh'], IBS_data_sex[idp_str+'_rh-lh'])
                d, ci_low, ci_high, tost_results_margin = equivalence_test(HC_data_sex[idp_str+'_rh-lh'], IBS_data_sex[idp_str+'_rh-lh'], margin=0.1)
                t_test_table.append({'Brain area':idp_str, 'Contrast': f'HC, {HC_sex} - IBS {group}, {IBS_sex}', 
                                     'Mean HC Right': mean_hc_right,
                                     'N HC':len(HC_data_sex[idp_str+'_rh-lh']),
                                     'N IBS':len(IBS_data_sex[idp_str+'_rh-lh']),
                                        'Std HC Right': std_hc_right,
                                        'Mean HC Left': mean_hc_left,
                                        'Std HC Left': std_hc_left,
                                        'Mean IBS Right': mean_ibs_right,
                                        'Std IBS Right': std_ibs_right,
                                        'Mean IBS Left': mean_ibs_left,
                                        'Std IBS Left': std_ibs_left,
                                        'Mean HC diff': mean_hc_diff,
                                        'Std HC diff': std_hc_diff,
                                        'Mean IBS diff': mean_ibs_diff,
                                        'Std IBS diff': std_ibs_diff,
                                        'df': df,
                                        'ci_low': ci_low, 'ci_high': ci_high, 
                                        'effect_size': d, 
                                        't_stat': t_stat, 'p_value': p_value})

t_test_table = pd.DataFrame(t_test_table)
adjusted_p_values = multipletests(t_test_table['p_value'], method='fdr_bh')[1]
t_test_table['FDR_p'] = adjusted_p_values

## Table display

In [5]:
# Modify the Contrast column to replace numbers with strings based on their position
def replace_contrast(contrast):
    # Split the string into parts
    parts = contrast.split(' - ')
    
    # Process each part
    for i in range(len(parts)):
        # Split by comma to handle the two segments
        sub_parts = parts[i].split(', ')
        
        # Replace the first part (before the comma)
        if sub_parts[0] == 'HC':
            sub_parts[0] = 'HC'
        elif sub_parts[0] == 'IBS 1':
            sub_parts[0] = 'IBS ROME'
        elif sub_parts[0] == 'IBS 3':
            sub_parts[0] = 'IBS both'
        
        # Replace the second part (after the comma)
        if len(sub_parts) > 1:
            if sub_parts[1] == '0':
                sub_parts[1] = 'Female'
            elif sub_parts[1] == '1':
                sub_parts[1] = 'Male'
        
        # Join the sub_parts back together
        parts[i] = ', '.join(sub_parts)
    
    # Join the main parts back together
    return ' - '.join(parts)

# Apply the function to the Contrast column
t_test_table['Contrast'] = t_test_table['Contrast'].apply(replace_contrast)

In [6]:
# Function to format Mean and Std into "Mean (SD)"
def format_mean_std(mean, std):
    return f"{mean:.3f} ({std:.3f})"

# Create new columns for Mean and Std values in the desired format
t_test_table['N HC'] = t_test_table['N HC'].apply(lambda x: f"{x:.0f}")
t_test_table['N IBS'] = t_test_table['N IBS'].apply(lambda x: f"{x:.0f}")
t_test_table['Mean HC Right (SD)'] = t_test_table.apply(lambda row: format_mean_std(row['Mean HC Right'], row['Std HC Right']), axis=1)
t_test_table['Mean HC Left (SD)'] = t_test_table.apply(lambda row: format_mean_std(row['Mean HC Left'], row['Std HC Left']), axis=1)
t_test_table['Mean IBS Right (SD)'] = t_test_table.apply(lambda row: format_mean_std(row['Mean IBS Right'], row['Std IBS Right']), axis=1)
t_test_table['Mean IBS Left (SD)'] = t_test_table.apply(lambda row: format_mean_std(row['Mean IBS Left'], row['Std IBS Left']), axis=1)
t_test_table['Mean HC Diff (SD)'] = t_test_table.apply(lambda row: format_mean_std(row['Mean HC diff'], row['Std HC diff']), axis=1)
t_test_table['Mean IBS Diff (SD)'] = t_test_table.apply(lambda row: format_mean_std(row['Mean IBS diff'], row['Std IBS diff']), axis=1)
t_test_table['ci_low'] = t_test_table['ci_low'].apply(lambda x: f"{x:.3f}")
t_test_table['ci_high'] = t_test_table['ci_high'].apply(lambda x: f"{x:.3f}")
t_test_table['effect_size'] = t_test_table['effect_size'].apply(lambda x: f"{x:.3f}")
t_test_table['df'] = t_test_table['df'].apply(lambda x: f"{x:.0f}")
t_test_table['t_stat'] = t_test_table['t_stat'].apply(lambda x: f"{x:.3f}")
t_test_table['p_value'] = t_test_table['p_value'].apply(lambda x: f"{x:.3f}")
t_test_table['FDR_p'] = t_test_table['FDR_p'].apply(lambda x: f"{x:.3f}")
# Optionally, drop the original Mean and Std columns if no longer needed
t_test_table.drop(columns=['Mean HC Left', 'Std HC Left', 'Mean HC Right', 'Std HC Right', 
                                'Mean IBS Left', 'Std IBS Left', 'Mean IBS Right', 'Std IBS Right', 
                                'Mean HC diff', 'Std HC diff', 'Mean IBS diff', 'Std IBS diff'], inplace=True)

# Display the updated DataFrame
t_test_table.to_csv(os.path.join(out_dir, f'interaction_analysis_contrast_rh-lh_draft.csv'), index=False)

# Equivalence test

In [7]:
from statsmodels.stats.multitest import multipletests
from utils_norm.utils_analyses import equivalence_test

def average_hemisphere_deviation(df):
    # Extract deviation columns (assuming 'eid' or similar is preserved)
    lh_cols = [col for col in df.columns if col.startswith('lh_')]
    rh_cols = [col for col in df.columns if col.startswith('rh_')]

    # Match ROIs by stripping the prefix
    roi_names = [col.replace('lh_', '') for col in lh_cols if col.replace('lh_', '') in [r.replace('rh_', '') for r in rh_cols]]

    avg_deviation = pd.DataFrame()
    avg_deviation['eid'] = df['eid']

    for roi in roi_names:
        lh_col = f'lh_{roi}'
        rh_col = f'rh_{roi}'
        avg_deviation[roi] = df[[lh_col, rh_col]].mean(axis=1)

    return avg_deviation, roi_names

In [8]:
for file_name in ['CT', 'SA', 'CV']:
    results_dir = os.path.join(root_dir, '3_rerun_whole_work', '4_eq_test')
    os.makedirs(results_dir,exist_ok=True)
    deviation_dir = os.path.join(model_dir, file_name+'_age_45_85', 'perm_'+str(perm))
    HC_deviation = pd.read_csv(os.path.join(deviation_dir, 'HC_deviation.csv'))
    IBS_deviation = pd.read_csv(os.path.join(deviation_dir, 'Patient_deviation_IBS.csv'))
    HC_avg, rois = average_hemisphere_deviation(HC_deviation)
    IBS_avg, rois = average_hemisphere_deviation(IBS_deviation)

    deviation_eq = pd.DataFrame(columns=['roi','effect_size','ci_low','ci_high',
                                            'tost_results_power equivalence','tost_p_fdr','tost_results_margin p_low', 'tost_results_margin p_high'])
    for idp_num, idp_str in enumerate(rois):
        d, ci_low, ci_high, tost_results_margin = equivalence_test(IBS_avg[idp_str], HC_avg[idp_str], margin=0.1)
        deviation_eq.loc[idp_num, 'roi'] = idp_str
        deviation_eq.loc[idp_num, 'effect_size'] = d
        deviation_eq.loc[idp_num, 'ci_low'] = ci_low
        deviation_eq.loc[idp_num, 'ci_high'] = ci_high
        deviation_eq.loc[idp_num, 'tost_results_margin p_low'] = tost_results_margin[1][1]
        deviation_eq.loc[idp_num, 'tost_results_margin p_high'] = tost_results_margin[2][1]
        deviation_eq.loc[idp_num, 'tost_results_power equivalence'] = tost_results_margin[0]

    deviation_ttest_fdr = multipletests(deviation_eq['tost_results_power equivalence'], alpha=0.05, method='fdr_bh')
    deviation_eq['tost_p_fdr'] = deviation_ttest_fdr[1]
    deviation_eq.to_csv(os.path.join(results_dir,file_name+'.csv'), index=False)